# Brain Tumor MRI Dataset Preparation

This notebook covers the full dataset preparation pipeline for the Brain Tumor MRI Classification project.

**Dataset:** [Brain Tumor MRI Dataset](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset) by masoudnickparvar  
**Classes:** glioma, meningioma, pituitary, no_tumor  
**Framework:** PyTorch + torchvision

---

## Steps Covered
1. Kaggle API setup and dataset download
2. Directory structure verification
3. Class distribution analysis
4. Sample image visualization
5. Image size verification
6. Train / Validation / Test split summary

## Step 1: Install Dependencies & Set Up Kaggle API

Upload your `kaggle.json` API credentials file before running this cell.  
You can download it from: **Kaggle → Account → Create New API Token**

In [ ]:
# Install Kaggle CLI
!pip install kaggle --quiet

# Upload kaggle.json (run this in Colab)
import os

# Option A: Upload via Colab file uploader
# from google.colab import files
# files.upload()  # upload kaggle.json

# Option B: If using Google Drive (mount first)
# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json

# Create .kaggle directory and set permissions
!mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API configured successfully.")

## Step 2: Download the Brain Tumor MRI Dataset

The dataset contains MRI images organized into two splits:
- `Training/` — used for model training and validation
- `Testing/` — held-out set for final evaluation

Each split has 4 subdirectories (one per class).

In [ ]:
import os

DATA_ROOT = "/content/brain_tumor_dataset"
os.makedirs(DATA_ROOT, exist_ok=True)

# Download and unzip the dataset
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p {DATA_ROOT} --unzip

print(f"\nDataset downloaded to: {DATA_ROOT}")
!ls {DATA_ROOT}

## Step 3: Verify Directory Structure

Confirm the expected folder layout before loading data.

In [ ]:
import os

DATA_ROOT = "/content/brain_tumor_dataset"
TRAIN_DIR = os.path.join(DATA_ROOT, "Training")
TEST_DIR  = os.path.join(DATA_ROOT, "Testing")

print("=== Directory Structure ===")
for split_name, split_dir in [("Training", TRAIN_DIR), ("Testing", TEST_DIR)]:
    print(f"\n{split_name}/")
    if os.path.exists(split_dir):
        for cls in sorted(os.listdir(split_dir)):
            cls_path = os.path.join(split_dir, cls)
            if os.path.isdir(cls_path):
                count = len([
                    f for f in os.listdir(cls_path)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
                ])
                print(f"  ├── {cls}/   ({count} images)")
    else:
        print(f"  [NOT FOUND] Expected at: {split_dir}")

## Step 4: Load Dataset with torchvision ImageFolder

`ImageFolder` automatically assigns integer labels based on subdirectory names (sorted alphabetically):
- `glioma` → 0
- `meningioma` → 1
- `no_tumor` → 2
- `pituitary` → 3

In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Basic transform for exploration (no augmentation here)
base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# Load full training set
full_train_dataset = datasets.ImageFolder(root=TRAIN_DIR, transform=base_transform)
test_dataset       = datasets.ImageFolder(root=TEST_DIR,  transform=base_transform)

CLASS_NAMES = full_train_dataset.classes
print(f"Classes detected: {CLASS_NAMES}")
print(f"Class → index mapping: {full_train_dataset.class_to_idx}")
print(f"\nTotal training images : {len(full_train_dataset)}")
print(f"Total test images     : {len(test_dataset)}")

## Step 5: Class Distribution Analysis

Visualize how many samples exist per class to detect any class imbalance.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

# Count per class in each split
train_labels = [label for _, label in full_train_dataset.samples]
test_labels  = [label for _, label in test_dataset.samples]

train_counts = Counter(train_labels)
test_counts  = Counter(test_labels)

train_class_counts = [train_counts[i] for i in range(len(CLASS_NAMES))]
test_class_counts  = [test_counts[i]  for i in range(len(CLASS_NAMES))]

print("Training set class distribution:")
for cls, cnt in zip(CLASS_NAMES, train_class_counts):
    print(f"  {cls:<12} : {cnt:>5} images")

print("\nTest set class distribution:")
for cls, cnt in zip(CLASS_NAMES, test_class_counts):
    print(f"  {cls:<12} : {cnt:>5} images")

# --- Bar Chart ---
x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, train_class_counts, width, label='Train', color='steelblue', edgecolor='black')
bars2 = ax.bar(x + width/2, test_class_counts,  width, label='Test',  color='coral',     edgecolor='black')

ax.set_xlabel('Tumor Class', fontsize=13)
ax.set_ylabel('Number of Images', fontsize=13)
ax.set_title('Brain Tumor MRI — Class Distribution', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, fontsize=12)
ax.legend(fontsize=11)
ax.bar_label(bars1, padding=3, fontsize=10)
ax.bar_label(bars2, padding=3, fontsize=10)
ax.set_ylim(0, max(train_class_counts) * 1.2)

plt.tight_layout()
plt.savefig('/content/class_distribution.png', dpi=150)
plt.show()
print("Chart saved to /content/class_distribution.png")

## Step 6: Visualize Sample Images from Each Class

Display a 4×4 grid — 4 random images per tumor class — to get a qualitative feel for the data.

In [ ]:
import random
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image

SAMPLES_PER_CLASS = 4
fig = plt.figure(figsize=(14, 12))
fig.suptitle('Sample MRI Images — 4 per Class', fontsize=16, fontweight='bold', y=1.01)

for row_idx, cls_name in enumerate(CLASS_NAMES):
    cls_dir = os.path.join(TRAIN_DIR, cls_name)
    all_images = [
        f for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    sampled = random.sample(all_images, min(SAMPLES_PER_CLASS, len(all_images)))

    for col_idx, img_file in enumerate(sampled):
        ax = fig.add_subplot(len(CLASS_NAMES), SAMPLES_PER_CLASS,
                             row_idx * SAMPLES_PER_CLASS + col_idx + 1)
        img_path = os.path.join(cls_dir, img_file)
        img = Image.open(img_path).convert('RGB')
        ax.imshow(img, cmap='gray' if img.mode == 'L' else None)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(cls_name, fontsize=12, rotation=90,
                          labelpad=10, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/sample_images_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print("Grid saved to /content/sample_images_grid.png")

## Step 7: Verify Image Sizes

Check whether all images share the same dimensions, or if resizing is needed during preprocessing.

In [ ]:
from PIL import Image
from collections import defaultdict

size_counter = defaultdict(int)
total_checked = 0
SAMPLE_LIMIT = 500  # check first 500 images for speed

for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(TRAIN_DIR, cls_name)
    imgs = [
        f for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    for img_file in imgs[:SAMPLE_LIMIT // len(CLASS_NAMES)]:
        try:
            with Image.open(os.path.join(cls_dir, img_file)) as im:
                size_counter[im.size] += 1
                total_checked += 1
        except Exception as e:
            print(f"Error reading {img_file}: {e}")

print(f"Checked {total_checked} images.")
print("\nTop image sizes (W x H):")
for size, count in sorted(size_counter.items(), key=lambda x: -x[1])[:10]:
    print(f"  {size[0]}x{size[1]}  →  {count} images")

if len(size_counter) == 1:
    print("\nAll images have the SAME size — no custom resizing logic needed (torchvision Resize handles it).")
else:
    print(f"\nImages have {len(size_counter)} different sizes — torchvision Resize(224, 224) will normalize them.")

## Step 8: Train / Validation / Test Split Summary

We will use the following split strategy:
- **Training set** (from `Training/`): 80% train, 20% validation (random split)
- **Test set** (from `Testing/`): held out entirely for final evaluation

In [ ]:
import torch
from torch.utils.data import random_split

# Reproducibility
SEED = 42
torch.manual_seed(SEED)

total_train = len(full_train_dataset)
val_size    = int(0.2 * total_train)
train_size  = total_train - val_size

train_subset, val_subset = random_split(
    full_train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(SEED)
)

total_all = train_size + val_size + len(test_dataset)

print("=" * 45)
print(f"{'Split':<20} {'Images':>8}  {'% of Total':>10}")
print("=" * 45)
print(f"{'Train':<20} {train_size:>8}  {100*train_size/total_all:>9.1f}%")
print(f"{'Validation':<20} {val_size:>8}  {100*val_size/total_all:>9.1f}%")
print(f"{'Test':<20} {len(test_dataset):>8}  {100*len(test_dataset)/total_all:>9.1f}%")
print("=" * 45)
print(f"{'TOTAL':<20} {total_all:>8}  {'100.0%':>10}")

print(f"\nSeed used for split: {SEED}")
print("These split indices will be reproduced identically in notebook 02 (training).")

## Step 9: Summary

Dataset preparation is complete. Key findings:

| Item | Value |
|------|-------|
| Classes | glioma, meningioma, no_tumor, pituitary |
| Input size | 224 × 224 (after resize) |
| Augmentation | Applied during training (see notebook 02) |
| Split strategy | 80/20 train-val from `Training/`, separate `Testing/` |
| Random seed | 42 |

**Next step:** Proceed to `02_model_training.ipynb` to train ResNet18 on this data.